<a href="https://colab.research.google.com/github/GabrielaRguezCampos/other/blob/main/FIFA_MetLife_Market_Basket_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # Market Basket Analysis
 This script identifies frequently bought-together items from transactional data


## Import libraries

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
from itertools import combinations
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

## Upload Datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pandas as pd

BASE_DIR = "/content/drive/MyDrive/HEAT x FIU - Temp Folder/Market Basket Analysis"

file_map = {
    "line_items_df":   "dnc_OrderLineItem.csv",
}

for var, fname in file_map.items():
    path = os.path.join(BASE_DIR, fname)
    df = pd.read_csv(path, low_memory=False)
    globals()[var] = df
    print(f"Loaded {var}: {df.shape}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded line_items_df: (633149, 38)


## Show

In [ ]:
line_items_df.head()

,LineItemKey,LocalCreatedAtDateKey,SeasonKey,OrderKey,OrderId,ItemKey,ItemName,ItemDescription,VendorCode,LocationKey,...,CreatedAtDateTimeVal,HCS_Key,dwh_insert_datetime,dwh_update_datetime,MasterEventKey,LocalCreatedAtTimeKey,LocalClosedAtDateKey,LocalClosedAtTimeKey,LocalClosedAtDateTimeVal,ItemFamilyGroupName
0,1,20250615,2025,35,356ba404-3b2b-4c2f-9bc3-3395964daef5,4,Visor,NaN,NaN,18,...,2025-06-15 19:01:51,0,2025-07-17 13:22:23.727000000,NaN,1,150100,20250615,150100,2025-06-15 15:01:59,Other
1,2,20250615,2025,444,51a3c872-187d-46da-9a6e-50342379de2d,4,Visor,NaN,NaN,18,...,2025-06-15 19:08:44,0,2025-07-17 13:22:23.727000000,NaN,1,150800,20250615,150900,2025-06-15 15:09:02,Other
2,3,20250615,2025,6,81e514b3-6f37-4232-81ee-dfa327ba266f,5,Employee Meal Voucher,NaN,NaN,33,...,2025-06-15 18:35:18,0,2025-07-17 13:22:23.727000000,NaN,1,143500,20250615,143500,2025-06-15 14:35:26,Other
3,4,20250615,2025,41,e4ebd660-006c-4581-9309-dbef92dd51ad,5,Employee Meal Voucher,NaN,NaN,33,...,2025-06-15 18:57:26,0,2025-07-17 13:22:23.727000000,NaN,1,145700,20250615,145800,2025-06-15 14:58:10,Other
4,5,20250615,2025,46,a4130eca-688d-4882-aeba-7281b7f28f55,5,Employee Meal Voucher,NaN,NaN,33,...,2025-06-15 18:34:37,0,2025-07-17 13:22:23.727000000,NaN,1,143400,20250615,143400,2025-06-15 14:34:41,Other


In [ ]:
#  Inspect columns to infer keys
print('line_items columns:')
print(list(line_items_df.columns))

line_items columns:
['LineItemKey', 'LocalCreatedAtDateKey', 'SeasonKey', 'OrderKey', 'OrderId', 'ItemKey', 'ItemName', 'ItemDescription', 'VendorCode', 'LocationKey', 'LocationName', 'ItemCategoryName', 'SectionKey', 'SectionName', 'SectionCategory', 'SectionLevel', 'SectionGroup', 'Quantity', 'Price', 'Cost', 'RefundedFlag', 'VoidedFlag', 'Comments', 'IsTaxExempt', 'UnitOfMeasure', 'Units', 'CreatedAtDateKey', 'CreatedAtTimeKey', 'CreatedAtDateTimeVal', 'HCS_Key', 'dwh_insert_datetime', 'dwh_update_datetime', 'MasterEventKey', 'LocalCreatedAtTimeKey', 'LocalClosedAtDateKey', 'LocalClosedAtTimeKey', 'LocalClosedAtDateTimeVal', 'ItemFamilyGroupName']


## Clean

In [ ]:
print(f"\nSTEP 2: Data cleaning...")

initial_count = len(df)

# Remove rows with missing essential data
df = df[df['OrderKey'].notna() & df['ItemKey'].notna() & df['ItemName'].notna()]
print(f"✓ Removed rows with missing keys/names: {initial_count - len(df):,} rows removed")

# Convert Quantity to numeric and filter positive quantities
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
before_qty = len(df)
df = df[df['Quantity'] > 0]
print(f"✓ Kept positive quantities only: {before_qty - len(df):,} rows removed")

# Handle RefundedFlag - exclude refunded items
before_refund = len(df)
if 'RefundedFlag' in df.columns:
    # Handle different possible formats for RefundedFlag
    if df['RefundedFlag'].dtype == bool:
        df = df[~df['RefundedFlag']]
    else:
        # Try numeric (0 = not refunded, 1 = refunded)
        try:
            refund_vals = pd.to_numeric(df['RefundedFlag'], errors='coerce')
            df = df[(refund_vals.isna()) | (refund_vals == 0)]
        except:
            # Try string interpretation
            refund_strings = df['RefundedFlag'].astype(str).str.lower()
            df = df[~refund_strings.isin(['true', '1', 'yes', 'y'])]
    print(f"✓ Excluded refunded items: {before_refund - len(df):,} rows removed")


STEP 2: Data cleaning...
✓ Removed rows with missing keys/names: 0 rows removed
✓ Kept positive quantities only: 0 rows removed
✓ Excluded refunded items: 0 rows removed


In [ ]:
# Handle VoidedFlag - exclude voided items
before_void = len(df)
if 'VoidedFlag' in df.columns:
    if df['VoidedFlag'].dtype == bool:
        df = df[~df['VoidedFlag']]
    else:
        try:
            void_vals = pd.to_numeric(df['VoidedFlag'], errors='coerce')
            df = df[(void_vals.isna()) | (void_vals == 0)]
        except:
            void_strings = df['VoidedFlag'].astype(str).str.lower()
            df = df[~void_strings.isin(['true', '1', 'yes', 'y'])]
    print(f"✓ Excluded voided items: {before_void - len(df):,} rows removed")

print(f"✓ Clean dataset: {len(df):,} rows remaining")

✓ Excluded voided items: 0 rows removed
✓ Clean dataset: 633,149 rows remaining


##  CREATE TRANSACTION-ITEM PAIRS

In [ ]:
# STEP 3: CREATE TRANSACTION DATA (PROPERLY DEDUPLICATED)
# ============================================================================
print(f"\nSTEP 3: Creating properly deduplicated transactions...")

# Clean item names
df['ItemName'] = df['ItemName'].astype(str).str.strip()
df = df[df['ItemName'] != '']

# CRITICAL FIX: Create unique order-item combinations
# This removes the duplication issue that caused 633K "unique" items
transactions = df[['OrderKey', 'ItemName']].drop_duplicates()

print(f"✓ Raw line items: {len(df):,}")
print(f"✓ Unique order-item combinations: {len(transactions):,}")

# Rename columns for clarity
transactions.columns = ['order_id', 'item']


STEP 3: Creating properly deduplicated transactions...
✓ Raw line items: 633,149
✓ Unique order-item combinations: 633,148


In [ ]:

total_orders = transactions['order_id'].nunique()
unique_items = transactions['item'].nunique()

print(f"✅ CORRECTED COUNTS:")
print(f"Total unique orders: {total_orders:,}")
print(f"Unique items: {unique_items:,}")

# Analyze order patterns
order_sizes = transactions.groupby('order_id')['item'].nunique()
orders_with_multiple = (order_sizes >= 2).sum()
avg_items_per_order = order_sizes.mean()
median_items_per_order = order_sizes.median()

print(f"Orders with 2+ items: {orders_with_multiple:,}")
print(f"Average items per order: {avg_items_per_order:.2f}")
print(f"Median items per order: {median_items_per_order:.1f}")

# Show order size distribution
print(f"\n📊 Order size distribution:")
size_dist = order_sizes.value_counts().sort_index().head(10)
for size, count in size_dist.items():
    print(f"  {size:2d} items: {count:,} orders")

print(f"\n🎯 These numbers should now match Julius AI's results!")


STEP 4: Validation counts...
✅ CORRECTED COUNTS:
Total unique orders: 366,471
Unique items: 419
Orders with 2+ items: 167,307
Average items per order: 1.73
Median items per order: 1.0

📊 Order size distribution:
   1 items: 199,164 orders
   2 items: 101,587 orders
   3 items: 42,253 orders
   4 items: 16,151 orders
   5 items: 5,238 orders
   6 items: 1,499 orders
   7 items: 430 orders
   8 items: 101 orders
   9 items: 39 orders
  10 items: 5 orders

🎯 These numbers should now match Julius AI's results!


In [ ]:
print("Creating transaction-item pairs...")

# Create the basic transaction data
transactions = working_df[[order_col, 'item_display']].drop_duplicates()
transactions.columns = ['order_id', 'item']

print(f"✓ Unique order-item pairs: {len(transactions):,}")

# Analyze order patterns
orders_summary = transactions.groupby('order_id')['item'].nunique()
print(f"✓ Total unique orders: {len(orders_summary):,}")
print(f"✓ Average items per order: {orders_summary.mean():.2f}")
print(f"✓ Median items per order: {orders_summary.median():.2f}")
print(f"✓ Max items in single order: {orders_summary.max()}")



Creating transaction-item pairs...
✓ Unique order-item pairs: 633,149
✓ Total unique orders: 366,471
✓ Average items per order: 1.73
✓ Median items per order: 1.00
✓ Max items in single order: 11


## Generate Item Pairs ad calculate metrics

In [ ]:
# STEP 5: GENERATE ITEM PAIRS
# ============================================================================
print(f"\nSTEP 5: Generating item pairs...")

pair_counter = Counter()
item_counter = Counter()
qualifying_orders = 0

print("Processing orders to find item pairs...")
for order_id, group in tqdm(transactions.groupby('order_id'), desc="Processing orders"):
    items = list(group['item'].unique())
    items = sorted(set(items))  # Remove any duplicates and sort

    # Count individual item occurrences
    for item in items:
        item_counter[item] += 1

    # Generate pairs for orders with 2+ items
    if len(items) >= 2:
        qualifying_orders += 1
        for item_a, item_b in combinations(items, 2):
            # Store pairs in consistent order to avoid duplicates
            pair = tuple(sorted([item_a, item_b]))
            pair_counter[pair] += 1

print(f"✅ PAIR GENERATION COMPLETE:")
print(f"Orders with 2+ items: {qualifying_orders:,}")
print(f"Unique items found: {len(item_counter):,}")
print(f"Unique pairs generated: {len(pair_counter):,}")


STEP 5: Generating item pairs...
Processing orders to find item pairs...


Processing orders: 100%|██████████| 366471/366471 [00:38<00:00, 9584.50it/s]


✅ PAIR GENERATION COMPLETE:
Orders with 2+ items: 167,307
Unique items found: 419
Unique pairs generated: 7,253


## Calculate Market Basket Metrics

In [ ]:
# ============================================================================
# STEP 6: CALCULATE MARKET BASKET METRICS
# ============================================================================
print(f"\nSTEP 6: Calculating market basket metrics...")

if len(pair_counter) == 0:
    print("❌ No pairs found - all orders contain only single items")
    pairs_df = pd.DataFrame()
else:
    pairs_data = []

    for (item_a, item_b), pair_count in pair_counter.items():
        # Basic counts
        item_a_count = item_counter[item_a]
        item_b_count = item_counter[item_b]

        # Support: How often items appear together
        support = pair_count / qualifying_orders if qualifying_orders > 0 else 0

        # Confidence: Given A, probability of B (and vice versa)
        confidence_a_to_b = pair_count / item_a_count if item_a_count > 0 else 0
        confidence_b_to_a = pair_count / item_b_count if item_b_count > 0 else 0

        # Lift: How much more likely than independent occurrence
        prob_a = item_a_count / qualifying_orders if qualifying_orders > 0 else 0
        prob_b = item_b_count / qualifying_orders if qualifying_orders > 0 else 0
        expected_together = prob_a * prob_b
        lift = support / expected_together if expected_together > 0 else np.nan

        pairs_data.append([
            item_a, item_b, pair_count, support,
            item_a_count, item_b_count,
            confidence_a_to_b, confidence_b_to_a, lift
        ])

    # Create DataFrame with results
    pairs_df = pd.DataFrame(pairs_data, columns=[
        'item_a', 'item_b', 'pair_count', 'support',
        'item_a_count', 'item_b_count',
        'confidence_a_to_b', 'confidence_b_to_a', 'lift'
    ])

    # Sort by frequency and lift
    pairs_df = pairs_df.sort_values(['pair_count', 'lift'], ascending=[False, False]).reset_index(drop=True)

    print(f"✓ Generated metrics for {len(pairs_df):,} item pairs")



STEP 6: Calculating market basket metrics...
✓ Generated metrics for 7,253 item pairs


## DISPLAY RESULTS

In [ ]:
print(f"\n" + "="*80)
print("🏆 TOP 20 FREQUENTLY BOUGHT TOGETHER ITEMS")
print("="*80)

if not pairs_df.empty:
    top_pairs = pairs_df.head(20)

    for i, (_, row) in enumerate(top_pairs.iterrows(), 1):
        print(f"\n{i:2d}. {row['item_a']}")
        print(f"    + {row['item_b']}")
        print(f"    📈 Bought together: {row['pair_count']:,} times")
        print(f"    📊 Support: {row['support']:.4f} | Confidence: {max(row['confidence_a_to_b'], row['confidence_b_to_a']):.3f} | Lift: {row['lift']:.2f}")

    # Summary statistics
    print(f"\n📈 SUMMARY STATISTICS:")
    print(f"Most frequent pair: {pairs_df.iloc[0]['pair_count']:,} occurrences")
    print(f"Average pair frequency: {pairs_df['pair_count'].mean():.1f}")
    print(f"Pairs with lift > 1.0: {(pairs_df['lift'] > 1.0).sum():,}")
    print(f"Pairs with lift > 2.0: {(pairs_df['lift'] > 2.0).sum():,}")


🏆 TOP 20 FREQUENTLY BOUGHT TOGETHER ITEMS

 1. Medium Pepsi Cup
    + Pepsi
    📈 Bought together: 24,055 times
    📊 Support: 0.1438 | Confidence: 0.957 | Lift: 3.57

 2. Aquafina Bottled Water 20oz
    + Medium Pepsi Cup
    📈 Bought together: 14,860 times
    📊 Support: 0.0888 | Confidence: 0.332 | Lift: 0.41

 3. 4pc Chicken Tender
    + Aquafina Bottled Water 20oz
    📈 Bought together: 8,406 times
    📊 Support: 0.0502 | Confidence: 0.347 | Lift: 0.43

 4. Aquafina Bottled Water 20oz
    + Pepsi
    📈 Bought together: 8,105 times
    📊 Support: 0.0484 | Confidence: 0.322 | Lift: 0.40

 5. Aquafina Bottled Water 20oz
    + Stadium Hot Dog
    📈 Bought together: 7,223 times
    📊 Support: 0.0432 | Confidence: 0.346 | Lift: 0.43

 6. Medium Pepsi Cup
    + Stadium Hot Dog
    📈 Bought together: 7,093 times
    📊 Support: 0.0424 | Confidence: 0.340 | Lift: 1.27

 7. 4pc Chicken Tender
    + Medium Pepsi Cup
    📈 Bought together: 6,370 times
    📊 Support: 0.0381 | Confidence: 0.263

## Save Results

In [ ]:
print(f"\n💾 SAVING RESULTS...")

BASE_DIR = "/content/drive/MyDrive/HEAT x FIU - Temp Folder/Market Basket Analysis"

# Save main results
results_file = f"{BASE_DIR}/line_items_only_results.csv"
pairs_df.to_csv(results_file, index=False)
print(f"✓ Saved results to: line_items_only_results.csv")

# Save transaction summary
trans_file = f"{BASE_DIR}/line_items_only_transactions.csv"
transaction_summary = transactions.groupby('order_id')['item'].apply(list).reset_index()
transaction_summary['item_count'] = transaction_summary['item'].apply(len)
transaction_summary.to_csv(trans_file, index=False)
print(f"✓ Saved transactions to: line_items_only_transactions.csv")

# Save item frequency analysis
item_freq = pd.DataFrame(item_counter.most_common(), columns=['item', 'frequency'])
item_freq['support'] = item_freq['frequency'] / qualifying_orders
freq_file = f"{BASE_DIR}/item_frequency_analysis.csv"
item_freq.to_csv(freq_file, index=False)
print(f"✓ Saved item frequency analysis to: item_frequency_analysis.csv")


💾 SAVING RESULTS...
✓ Saved results to: line_items_only_results.csv
✓ Saved transactions to: line_items_only_transactions.csv
✓ Saved item frequency analysis to: item_frequency_analysis.csv


## Summary

In [ ]:
print(f"\n" + "="*80)
print("🎉 CORRECTED ANALYSIS COMPLETE!")
print("="*80)
print(f"📊 Total orders analyzed: {total_orders:,}")
print(f"🛍️  Orders with multiple items: {qualifying_orders:,}")
print(f"📦 Unique products: {unique_items:,}")
print(f"🔗 Item pairs discovered: {len(pairs_df):,}")
print(f"💾 Results saved to Google Drive")


if not pairs_df.empty:
    print(f"\n🎯 BUSINESS INSIGHTS:")
    best_pair = pairs_df.iloc[0]
    print(f"Top pair: {best_pair['item_a']} + {best_pair['item_b']}")
    print(f"Bought together {best_pair['pair_count']:,} times with {best_pair['lift']:.2f}x lift")
    print(f"Ready for cross-selling, bundling, and promotional strategies!")


🎉 CORRECTED ANALYSIS COMPLETE!
📊 Total orders analyzed: 366,471
🛍️  Orders with multiple items: 167,307
📦 Unique products: 419
🔗 Item pairs discovered: 7,253
💾 Results saved to Google Drive

🎯 BUSINESS INSIGHTS:
Top pair: Medium Pepsi Cup + Pepsi
Bought together 24,055 times with 3.57x lift
Ready for cross-selling, bundling, and promotional strategies!


# Pepsi Cup analysis

In [ ]:
# ============================================================================
# PART 1: INVESTIGATE THE PEPSI CUP PRICING PATTERN
# ============================================================================
print("PART 1: Investigating 'Medium Pepsi Cup' pricing pattern...")

# Use your loaded line_items_df
df = line_items_df.copy()

# Look for the Medium Pepsi Cup specifically
pepsi_cup_mask = df['ItemName'].str.contains('Medium Pepsi Cup', case=False, na=False)
pepsi_cup_data = df[pepsi_cup_mask].copy()

print(f"✓ Found {len(pepsi_cup_data):,} 'Medium Pepsi Cup' line items")

if len(pepsi_cup_data) > 0:
    # Analyze the cup pricing
    print(f"\n📊 MEDIUM PEPSI CUP ANALYSIS:")
    print(f"Price statistics:")
    print(f"  Min price: ${pepsi_cup_data['Price'].min():.2f}")
    print(f"  Max price: ${pepsi_cup_data['Price'].max():.2f}")
    print(f"  Average price: ${pepsi_cup_data['Price'].mean():.2f}")
    print(f"  Most common price: ${pepsi_cup_data['Price'].mode().iloc[0]:.2f}")

    # Check price distribution
    price_dist = pepsi_cup_data['Price'].value_counts().head(10)
    print(f"\nPrice distribution:")
    for price, count in price_dist.items():
        print(f"  ${price:.2f}: {count:,} times")

    # Look for pepsi drinks in the same orders
    pepsi_cup_orders = set(pepsi_cup_data['OrderKey'].unique())
    print(f"\n🔍 ANALYZING ORDERS WITH PEPSI CUPS...")
    print(f"Orders containing Medium Pepsi Cup: {len(pepsi_cup_orders):,}")

    # Get all items from orders that contain pepsi cups
    orders_with_cups = df[df['OrderKey'].isin(pepsi_cup_orders)].copy()

    # Look for pepsi drinks (likely containing "pepsi" but not "cup")
    pepsi_drink_mask = (
        orders_with_cups['ItemName'].str.contains('pepsi', case=False, na=False) &
        ~orders_with_cups['ItemName'].str.contains('cup', case=False, na=False)
    )
    pepsi_drinks = orders_with_cups[pepsi_drink_mask]

    print(f"Potential pepsi drinks in these orders: {len(pepsi_drinks):,} line items")

    if len(pepsi_drinks) > 0:
        print(f"\n📊 PEPSI DRINK ANALYSIS:")
        print(f"Unique pepsi drink items: {pepsi_drinks['ItemName'].nunique()}")
        print(f"\nTop pepsi drink items:")
        drink_counts = pepsi_drinks['ItemName'].value_counts().head(10)
        for item, count in drink_counts.items():
            print(f"  {item}: {count:,} times")

        print(f"\nPepsi drink pricing:")
        print(f"  Min price: ${pepsi_drinks['Price'].min():.2f}")
        print(f"  Max price: ${pepsi_drinks['Price'].max():.2f}")
        print(f"  Average price: ${pepsi_drinks['Price'].mean():.2f}")

        # Check how many are $0
        zero_price_drinks = pepsi_drinks[pepsi_drinks['Price'] == 0]
        print(f"  Zero-price pepsi drinks: {len(zero_price_drinks):,} ({len(zero_price_drinks)/len(pepsi_drinks)*100:.1f}%)")

    # Sample some orders to see the pattern
    print(f"\n🔍 SAMPLE ORDERS WITH PEPSI CUPS:")
    sample_orders = list(pepsi_cup_orders)[:5]

    for i, order_key in enumerate(sample_orders, 1):
        order_items = orders_with_cups[orders_with_cups['OrderKey'] == order_key][
            ['ItemName', 'Price', 'Quantity']
        ].copy()
        print(f"\nSample Order {i} (OrderKey: {order_key}):")
        for _, row in order_items.iterrows():
            print(f"  {row['ItemName']}: ${row['Price']:.2f} x {row['Quantity']}")

else:
    print("❌ No 'Medium Pepsi Cup' items found in the data")

PART 1: Investigating 'Medium Pepsi Cup' pricing pattern...
✓ Found 44,806 'Medium Pepsi Cup' line items

📊 MEDIUM PEPSI CUP ANALYSIS:
Price statistics:
  Min price: $0.00
  Max price: $110.50
  Average price: $9.24
  Most common price: $6.50

Price distribution:
  $6.50: 30,690 times
  $13.00: 10,789 times
  $19.50: 2,308 times
  $26.00: 735 times
  $32.50: 194 times
  $39.00: 61 times
  $45.50: 18 times
  $52.00: 6 times
  $58.50: 2 times
  $110.50: 1 times

🔍 ANALYZING ORDERS WITH PEPSI CUPS...
Orders containing Medium Pepsi Cup: 44,806
Potential pepsi drinks in these orders: 32,194 line items

📊 PEPSI DRINK ANALYSIS:
Unique pepsi drink items: 3

Top pepsi drink items:
  Pepsi: 24,055 times
  Diet Pepsi: 4,534 times
  Pepsi Zero: 3,605 times

Pepsi drink pricing:
  Min price: $0.00
  Max price: $0.00
  Average price: $0.00
  Zero-price pepsi drinks: 32,194 (100.0%)

🔍 SAMPLE ORDERS WITH PEPSI CUPS:

Sample Order 1 (OrderKey: 131072):
  Pepsi: $0.00 x 1
  Medium Pepsi Cup: $6.50 x 1


In [ ]:
# ============================================================================
# PART 2: IDENTIFY ALL PROBLEMATIC CUP/CONTAINER ITEMS
# ============================================================================
print(f"\n" + "="*80)
print("PART 2: Identifying all cup/container items...")

# Look for cup-related items more broadly
cup_keywords = ['cup', 'container', 'bowl', 'plate', 'box', 'bag']
cup_items = []

for keyword in cup_keywords:
    mask = df['ItemName'].str.contains(keyword, case=False, na=False)
    items = df[mask]['ItemName'].unique()
    cup_items.extend(items)

cup_items = list(set(cup_items))  # Remove duplicates
print(f"✓ Found {len(cup_items)} potential container/cup items")

# Show the cup items and their frequency
cup_analysis = df[df['ItemName'].isin(cup_items)]['ItemName'].value_counts()
print(f"\nTop container/cup items:")
for item, count in cup_analysis.head(15).items():
    avg_price = df[df['ItemName'] == item]['Price'].mean()
    print(f"  {item}: {count:,} times (avg: ${avg_price:.2f})")


PART 2: Identifying all cup/container items...
✓ Found 20 potential container/cup items

Top container/cup items:
  Medium Pepsi Cup: 44,806 times (avg: $9.24)
  Frozen Yard Cup: 2,602 times (avg: $27.18)
  24oz Soda Cup: 1,390 times (avg: $10.79)
  Cheese Cup: 1,099 times (avg: $1.14)
  Large Pepsi Cup: 954 times (avg: $10.64)
  Asian Noodle Bowl w/Chicken: 718 times (avg: $14.81)
  Jets Souvenir Soda Cup: 641 times (avg: $11.26)
  Value Pepsi Cup: 544 times (avg: $4.17)
  Asian Noodle Bowl w/Pork: 230 times (avg: $14.13)
  Fruit Cup: 176 times (avg: $7.76)
  MF Ice Cup: 80 times (avg: $12.50)
  Asian Noodle Bowl: 75 times (avg: $14.91)
  Tajin Fruit Cup: 72 times (avg: $13.67)
  Quinoa Bowl: 34 times (avg: $10.59)
  Fruit Cup FIFA: 31 times (avg: $7.03)


In [ ]:
# ============================================================================
#PART 3: MARKET BASKET ANALYSIS EXCLUDING ALL CUP ITEMS
# ============================================================================
print(f"\n" + "="*80)
print("PART 3: Market Basket Analysis excluding cup items...")

# Define all cup items to filter out
cup_items_to_filter = [
    'Medium Pepsi Cup',
    'Frozen Yard Cup',
    '24oz Soda Cup',
    'Large Pepsi Cup',
    'Jets Souvenir Soda Cup',
    'Value Pepsi Cup'
]

print(f"Filtering out the following cup items:")
for cup in cup_items_to_filter:
    print(f"  • {cup}")

# Create filtered dataset excluding all specified cup items
df_filtered = df.copy()
initial_items = len(df)

# Filter out each cup item
for cup_item in cup_items_to_filter:
    before_filter = len(df_filtered)
    df_filtered = df_filtered[~df_filtered['ItemName'].str.contains(cup_item, case=False, na=False)]
    removed = before_filter - len(df_filtered)
    if removed > 0:
        print(f"✓ Removed {removed:,} '{cup_item}' line items")

total_removed = initial_items - len(df_filtered)
print(f"✓ Total cup items removed: {total_removed:,}")
print(f"✓ Working with {len(df_filtered):,} remaining line items")

# Apply the same cleaning as before
df_filtered = df_filtered[
    df_filtered['OrderKey'].notna() &
    df_filtered['ItemKey'].notna() &
    df_filtered['ItemName'].notna()
]

# Convert Quantity and filter positive
df_filtered['Quantity'] = pd.to_numeric(df_filtered['Quantity'], errors='coerce')
df_filtered = df_filtered[df_filtered['Quantity'] > 0]

# Handle RefundedFlag and VoidedFlag
for flag_col in ['RefundedFlag', 'VoidedFlag']:
    if flag_col in df_filtered.columns:
        if df_filtered[flag_col].dtype == bool:
            df_filtered = df_filtered[~df_filtered[flag_col]]
        else:
            try:
                flag_vals = pd.to_numeric(df_filtered[flag_col], errors='coerce')
                df_filtered = df_filtered[(flag_vals.isna()) | (flag_vals == 0)]
            except:
                flag_strings = df_filtered[flag_col].astype(str).str.lower()
                df_filtered = df_filtered[~flag_strings.isin(['true', '1', 'yes', 'y'])]

print(f"✓ Clean filtered dataset: {len(df_filtered):,} rows")

# Create deduplicated transactions (excluding pepsi cups)
df_filtered['ItemName'] = df_filtered['ItemName'].astype(str).str.strip()
df_filtered = df_filtered[df_filtered['ItemName'] != '']

transactions_filtered = df_filtered[['OrderKey', 'ItemName']].drop_duplicates()
transactions_filtered.columns = ['order_id', 'item']

print(f"✓ Filtered transactions: {len(transactions_filtered):,} order-item pairs")

# Validation counts for filtered data
total_orders_filtered = transactions_filtered['order_id'].nunique()
unique_items_filtered = transactions_filtered['item'].nunique()
order_sizes_filtered = transactions_filtered.groupby('order_id')['item'].nunique()
orders_with_multiple_filtered = (order_sizes_filtered >= 2).sum()

print(f"\n📊 FILTERED DATASET STATS:")
print(f"Total orders: {total_orders_filtered:,}")
print(f"Orders with 2+ items: {orders_with_multiple_filtered:,}")
print(f"Unique items: {unique_items_filtered:,}")


PART 3: Market Basket Analysis excluding cup items...
Filtering out the following cup items:
  • Medium Pepsi Cup
  • Frozen Yard Cup
  • 24oz Soda Cup
  • Large Pepsi Cup
  • Jets Souvenir Soda Cup
  • Value Pepsi Cup
✓ Removed 44,806 'Medium Pepsi Cup' line items
✓ Removed 2,602 'Frozen Yard Cup' line items
✓ Removed 1,390 '24oz Soda Cup' line items
✓ Removed 954 'Large Pepsi Cup' line items
✓ Removed 641 'Jets Souvenir Soda Cup' line items
✓ Removed 544 'Value Pepsi Cup' line items
✓ Total cup items removed: 50,937
✓ Working with 582,212 remaining line items
✓ Clean filtered dataset: 582,212 rows
✓ Filtered transactions: 582,211 order-item pairs

📊 FILTERED DATASET STATS:
Total orders: 364,431
Orders with 2+ items: 155,887
Unique items: 413


In [ ]:
# ============================================================================
# PART 4: GENERATE FILTERED PAIRS
# ============================================================================
print(f"\nGenerating item pairs (excluding Cups)...")

pair_counter_filtered = Counter()
item_counter_filtered = Counter()
qualifying_orders_filtered = 0

for order_id, group in tqdm(transactions_filtered.groupby('order_id'), desc="Processing filtered orders"):
    items = list(group['item'].unique())
    items = sorted(set(items))

    # Count individual items
    for item in items:
        item_counter_filtered[item] += 1

    # Generate pairs for orders with 2+ items
    if len(items) >= 2:
        qualifying_orders_filtered += 1
        for item_a, item_b in combinations(items, 2):
            pair = tuple(sorted([item_a, item_b]))
            pair_counter_filtered[pair] += 1

print(f"✓ Filtered pairs generated: {len(pair_counter_filtered):,}")



Generating item pairs (excluding Medium Pepsi Cup)...


Processing filtered orders: 100%|██████████| 364431/364431 [00:38<00:00, 9360.61it/s] 


✓ Filtered pairs generated: 6,872


In [ ]:
# ============================================================================
# PART 5: CALCULATE METRICS FOR FILTERED DATA
# ============================================================================
print(f"\nCalculating metrics for filtered data...")

if len(pair_counter_filtered) > 0:
    pairs_data_filtered = []

    for (item_a, item_b), pair_count in pair_counter_filtered.items():
        item_a_count = item_counter_filtered[item_a]
        item_b_count = item_counter_filtered[item_b]

        # Calculate metrics
        support = pair_count / qualifying_orders_filtered if qualifying_orders_filtered > 0 else 0
        confidence_a_to_b = pair_count / item_a_count if item_a_count > 0 else 0
        confidence_b_to_a = pair_count / item_b_count if item_b_count > 0 else 0

        prob_a = item_a_count / qualifying_orders_filtered if qualifying_orders_filtered > 0 else 0
        prob_b = item_b_count / qualifying_orders_filtered if qualifying_orders_filtered > 0 else 0
        expected = prob_a * prob_b
        lift = support / expected if expected > 0 else np.nan

        pairs_data_filtered.append([
            item_a, item_b, pair_count, support,
            item_a_count, item_b_count,
            confidence_a_to_b, confidence_b_to_a, lift
        ])

    pairs_df_filtered = pd.DataFrame(pairs_data_filtered, columns=[
        'item_a', 'item_b', 'pair_count', 'support',
        'item_a_count', 'item_b_count',
        'confidence_a_to_b', 'confidence_b_to_a', 'lift'
    ])

    pairs_df_filtered = pairs_df_filtered.sort_values(['pair_count', 'lift'], ascending=[False, False]).reset_index(drop=True)

    print(f"✓ Generated {len(pairs_df_filtered):,} filtered item pairs")


Calculating metrics for filtered data...
✓ Generated 6,872 filtered item pairs


In [ ]:
# ============================================================================
# PART 6: DISPLAY FILTERED RESULTS
# ============================================================================
print(f"\n" + "="*80)
print("🏆 TOP 20 PAIRS (EXCLUDING MEDIUM PEPSI CUP)")
print("="*80)

if not pairs_df_filtered.empty:
    top_filtered = pairs_df_filtered.head(20)

    for i, (_, row) in enumerate(top_filtered.iterrows(), 1):
        print(f"\n{i:2d}. {row['item_a']}")
        print(f"    + {row['item_b']}")
        print(f"    📈 Together: {row['pair_count']:,} times")
        print(f"    📊 Support: {row['support']:.4f} | Confidence: {max(row['confidence_a_to_b'], row['confidence_b_to_a']):.3f} | Lift: {row['lift']:.2f}")



🏆 TOP 20 PAIRS (EXCLUDING MEDIUM PEPSI CUP)

 1. 4pc Chicken Tender
    + Aquafina Bottled Water 20oz
    📈 Together: 8,406 times
    📊 Support: 0.0539 | Confidence: 0.347 | Lift: 0.40

 2. Aquafina Bottled Water 20oz
    + Pepsi
    📈 Together: 8,105 times
    📊 Support: 0.0520 | Confidence: 0.322 | Lift: 0.37

 3. Aquafina Bottled Water 20oz
    + Stadium Hot Dog
    📈 Together: 7,223 times
    📊 Support: 0.0463 | Confidence: 0.346 | Lift: 0.40

 4. Aquafina Bottled Water 20oz
    + Michelob Ultra Aluminum 16oz
    📈 Together: 5,610 times
    📊 Support: 0.0360 | Confidence: 0.233 | Lift: 0.27

 5. Pepsi
    + Stadium Hot Dog
    📈 Together: 4,211 times
    📊 Support: 0.0270 | Confidence: 0.202 | Lift: 1.25

 6. Aquafina Bottled Water 20oz
    + Pepsi Bottled Soda 20oz
    📈 Together: 4,089 times
    📊 Support: 0.0262 | Confidence: 0.301 | Lift: 0.35

 7. Aquafina Bottled Water 20oz
    + Bud Light Aluminum 16oz
    📈 Together: 3,990 times
    📊 Support: 0.0256 | Confidence: 0.266 | 

In [ ]:

# ============================================================================
# PART 8: SAVE FILTERED RESULTS
# ============================================================================
print(f"\n💾 SAVING FILTERED RESULTS...")

BASE_DIR = "/content/drive/MyDrive/HEAT x FIU - Temp Folder/Market Basket Analysis"

# Save filtered results
filtered_results_file = f"{BASE_DIR}/market_basket_filtered_no_pepsi_cup.csv"
pairs_df_filtered.to_csv(filtered_results_file, index=False)
print(f"✓ Saved filtered results to: market_basket_filtered_no_pepsi_cup.csv")

# Save pepsi cup analysis
if len(pepsi_cup_data) > 0:
    pepsi_analysis_file = f"{BASE_DIR}/pepsi_cup_pricing_analysis.csv"
    pepsi_cup_summary = pepsi_cup_data.groupby(['ItemName', 'Price']).agg({
        'OrderKey': 'count',
        'Quantity': 'sum'
    }).reset_index()
    pepsi_cup_summary.columns = ['ItemName', 'Price', 'LineItemCount', 'TotalQuantity']
    pepsi_cup_summary.to_csv(pepsi_analysis_file, index=False)
    print(f"✓ Saved pepsi cup analysis to: pepsi_cup_pricing_analysis.csv")

# Save container items analysis
container_summary = df[df['ItemName'].isin(cup_items)].groupby('ItemName').agg({
    'Price': ['mean', 'min', 'max'],
    'OrderKey': 'count'
}).reset_index()
container_summary.columns = ['ItemName', 'AvgPrice', 'MinPrice', 'MaxPrice', 'Frequency']
container_file = f"{BASE_DIR}/container_items_analysis.csv"
container_summary.to_csv(container_file, index=False)
print(f"✓ Saved container items analysis to: container_items_analysis.csv")

print(f"\n🎉 ANALYSIS COMPLETE!")
print(f"✅ Investigated Medium Pepsi Cup pricing pattern")
print(f"🔍 Identified {len(cup_items)} potential container items")
print(f"📊 Generated clean market basket results excluding problematic items")
print(f"💾 All results saved to Google Drive")

print(f"\n💡 KEY FINDINGS:")
if len(pepsi_cup_data) > 0:
    print(f"- Medium Pepsi Cup appears in {len(pepsi_cup_orders):,} orders")
    print(f"- Average cup price: ${pepsi_cup_data['Price'].mean():.2f}")
    if len(zero_price_drinks) > 0:
        print(f"- {len(zero_price_drinks):,} pepsi drinks are priced at $0.00")
        print("- This confirms the cup+drink bundling pattern!")
else:
    print("- No Medium Pepsi Cup pattern found in data")

print(f"\n🚀 NEXT STEPS:")
print("1. Review pepsi_cup_pricing_analysis.csv for detailed pricing patterns")
print("2. Use market_basket_filtered_no_pepsi_cup.csv for clean associations")
print("3. Consider filtering out other container items if needed")
print("4. Compare original vs filtered results to see the impact")


💾 SAVING FILTERED RESULTS...
✓ Saved filtered results to: market_basket_filtered_no_pepsi_cup.csv
✓ Saved pepsi cup analysis to: pepsi_cup_pricing_analysis.csv
✓ Saved container items analysis to: container_items_analysis.csv

🎉 ANALYSIS COMPLETE!
✅ Investigated Medium Pepsi Cup pricing pattern
🔍 Identified 20 potential container items
📊 Generated clean market basket results excluding problematic items
💾 All results saved to Google Drive

💡 KEY FINDINGS:
- Medium Pepsi Cup appears in 44,806 orders
- Average cup price: $9.24
- 32,194 pepsi drinks are priced at $0.00
- This confirms the cup+drink bundling pattern!

🚀 NEXT STEPS:
1. Review pepsi_cup_pricing_analysis.csv for detailed pricing patterns
2. Use market_basket_filtered_no_pepsi_cup.csv for clean associations
3. Consider filtering out other container items if needed
4. Compare original vs filtered results to see the impact


# Make outputs presentable and readable

In [ ]:
# Simple Business Tables Generator
# Creates two clean tables with business-friendly column names

import pandas as pd

print("=== CREATING BUSINESS-FRIENDLY TABLES ===\n")

BASE_DIR = "/content/drive/MyDrive/HEAT x FIU - Temp Folder/Market Basket Analysis"

# ============================================================================
# LOAD BOTH RESULT SETS
# ============================================================================
print("Loading results...")

# Load original results (with cups)
original_results = pd.read_csv(f"{BASE_DIR}/line_items_only_results.csv")
print(f"✓ Original results: {len(original_results):,} pairs")

# Load filtered results (without cups)
filtered_results = pd.read_csv(f"{BASE_DIR}/market_basket_filtered_no_pepsi_cup.csv")
print(f"✓ Filtered results: {len(filtered_results):,} pairs")

# ============================================================================
# CREATE BUSINESS-FRIENDLY TABLE 1: ORIGINAL RESULTS
# ============================================================================
print("\nCreating Table 1: Original Results...")

table1_original = original_results.copy()
table1_original = table1_original.rename(columns={
    'item_a': 'Product A',
    'item_b': 'Product B',
    'pair_count': 'Times Purchased Together',
    'support': 'Market Frequency',
    'confidence_a_to_b': 'A→B Confidence',
    'confidence_b_to_a': 'B→A Confidence',
    'lift': 'Association Strength',
    'item_a_count': 'Product A Total Sales',
    'item_b_count': 'Product B Total Sales'
})

# Format numbers for business presentation
table1_original['Times Purchased Together'] = table1_original['Times Purchased Together'].apply(lambda x: f"{x:,}")
table1_original['Market Frequency'] = table1_original['Market Frequency'].apply(lambda x: f"{x:.4f}")
table1_original['A→B Confidence'] = table1_original['A→B Confidence'].apply(lambda x: f"{x:.1%}")
table1_original['B→A Confidence'] = table1_original['B→A Confidence'].apply(lambda x: f"{x:.1%}")
table1_original['Association Strength'] = table1_original['Association Strength'].apply(lambda x: f"{x:.2f}x")
table1_original['Product A Total Sales'] = table1_original['Product A Total Sales'].apply(lambda x: f"{x:,}")
table1_original['Product B Total Sales'] = table1_original['Product B Total Sales'].apply(lambda x: f"{x:,}")

print(f"✓ Table 1 formatted: {len(table1_original):,} rows")

# ============================================================================
# CREATE BUSINESS-FRIENDLY TABLE 2: FILTERED RESULTS
# ============================================================================
print("Creating Table 2: Filtered Results...")

table2_filtered = filtered_results.copy()
table2_filtered = table2_filtered.rename(columns={
    'item_a': 'Product A',
    'item_b': 'Product B',
    'pair_count': 'Times Purchased Together',
    'support': 'Market Frequency',
    'confidence_a_to_b': 'A→B Confidence',
    'confidence_b_to_a': 'B→A Confidence',
    'lift': 'Association Strength',
    'item_a_count': 'Product A Total Sales',
    'item_b_count': 'Product B Total Sales'
})

# Format numbers for business presentation
table2_filtered['Times Purchased Together'] = table2_filtered['Times Purchased Together'].apply(lambda x: f"{x:,}")
table2_filtered['Market Frequency'] = table2_filtered['Market Frequency'].apply(lambda x: f"{x:.4f}")
table2_filtered['A→B Confidence'] = table2_filtered['A→B Confidence'].apply(lambda x: f"{x:.1%}")
table2_filtered['B→A Confidence'] = table2_filtered['B→A Confidence'].apply(lambda x: f"{x:.1%}")
table2_filtered['Association Strength'] = table2_filtered['Association Strength'].apply(lambda x: f"{x:.2f}x")
table2_filtered['Product A Total Sales'] = table2_filtered['Product A Total Sales'].apply(lambda x: f"{x:,}")
table2_filtered['Product B Total Sales'] = table2_filtered['Product B Total Sales'].apply(lambda x: f"{x:,}")

print(f"✓ Table 2 formatted: {len(table2_filtered):,} rows")

# ============================================================================
# SAVE BOTH TABLES
# ============================================================================
print("\nSaving business tables...")

# Save Table 1: Original Results
table1_file = f"{BASE_DIR}/Table1_Original_Results.csv"
table1_original.to_csv(table1_file, index=False)
print(f"✓ Saved Table 1 to: Table1_Original_Results.csv")

# Save Table 2: Filtered Results
table2_file = f"{BASE_DIR}/Table2_Filtered_Results.csv"
table2_filtered.to_csv(table2_file, index=False)
print(f"✓ Saved Table 2 to: Table2_Filtered_Results.csv")

# Save both tables in one Excel file with separate sheets
excel_file = f"{BASE_DIR}/Market_Basket_Results_Both_Tables.xlsx"
with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    table1_original.to_excel(writer, sheet_name='Original Results', index=False)
    table2_filtered.to_excel(writer, sheet_name='Filtered Results', index=False)

print(f"✓ Saved combined Excel file: Market_Basket_Results_Both_Tables.xlsx")

# ============================================================================
# SHOW PREVIEW OF BOTH TABLES
# ============================================================================
print(f"\n" + "="*80)
print("📊 TABLE 1: ORIGINAL RESULTS (Including Cups)")
print("="*80)
print(table1_original.head(10).to_string(index=False))

print(f"\n" + "="*80)
print("📊 TABLE 2: FILTERED RESULTS (Excluding Cups)")
print("="*80)
print(table2_filtered.head(10).to_string(index=False))

print(f"\n✅ COMPLETE!")
print(f"📁 Three files created:")
print(f"   • Table1_Original_Results.csv")
print(f"   • Table2_Filtered_Results.csv")
print(f"   • Market_Basket_Results_Both_Tables.xlsx (both sheets)")

# Show column definitions for customer reference
print(f"\n📋 COLUMN DEFINITIONS:")
print("• Product A & Product B: The two items frequently bought together")
print("• Times Purchased Together: Number of transactions containing both items")
print("• Market Frequency: How often this pair appears relative to all transactions")
print("• A to B Confidence: If customer buys Product A, probability of also buying Product B")
print("• B to A Confidence: If customer buys Product B, probability of also buying Product A")
print("• Association Strength: How much more likely items are bought together vs independently")
print("• Product A/B Total Sales: Total number of transactions each product appears in")

=== CREATING BUSINESS-FRIENDLY TABLES ===

Loading results...
✓ Original results: 7,253 pairs
✓ Filtered results: 6,872 pairs

Creating Table 1: Original Results...
✓ Table 1 formatted: 7,253 rows
Creating Table 2: Filtered Results...
✓ Table 2 formatted: 6,872 rows

Saving business tables...
✓ Saved Table 1 to: Table1_Original_Results.csv
✓ Saved Table 2 to: Table2_Filtered_Results.csv
✓ Saved combined Excel file: Market_Basket_Results_Both_Tables.xlsx

📊 TABLE 1: ORIGINAL RESULTS (Including Cups)
                  Product A                    Product B Times Purchased Together Market Frequency Product A Total Sales Product B Total Sales A→B Confidence B→A Confidence Association Strength
           Medium Pepsi Cup                        Pepsi                   24,055           0.1438                44,806                25,135          53.7%          95.7%                3.57x
Aquafina Bottled Water 20oz             Medium Pepsi Cup                   14,860           0.0888          

In [ ]:
# Simple Business Tables Generator
# Creates two clean tables with business-friendly column names

import pandas as pd

print("=== CREATING BUSINESS-FRIENDLY TABLES ===\n")

BASE_DIR = "/content/drive/MyDrive/HEAT x FIU - Temp Folder/Market Basket Analysis"

# ============================================================================
# LOAD BOTH RESULT SETS
# ============================================================================
print("Loading results...")

# Load original results (with cups)
original_results = pd.read_csv(f"{BASE_DIR}/line_items_only_results.csv")
print(f"✓ Original results: {len(original_results):,} pairs")

# Load filtered results (without cups)
filtered_results = pd.read_csv(f"{BASE_DIR}/market_basket_filtered_no_pepsi_cup.csv")
print(f"✓ Filtered results: {len(filtered_results):,} pairs")

# ============================================================================
# CREATE BUSINESS-FRIENDLY TABLE 1: ORIGINAL RESULTS
# ============================================================================
print("\nCreating Table 1: Original Results...")

table1_original = original_results.copy()
table1_original = table1_original.rename(columns={
    'item_a': 'Product A',
    'item_b': 'Product B',
    'pair_count': 'Times Purchased Together',
    'support': 'Market Frequency',
    'confidence_a_to_b': 'A→B Confidence',
    'confidence_b_to_a': 'B→A Confidence',
    'lift': 'Association Strength',
    'item_a_count': 'Product A Total Sales',
    'item_b_count': 'Product B Total Sales'
})

# Format numbers for business presentation
table1_original['Times Purchased Together'] = table1_original['Times Purchased Together'].apply(lambda x: f"{x:,}")
table1_original['Market Frequency'] = table1_original['Market Frequency'].apply(lambda x: f"{x:.4f}")
table1_original['A→B Confidence'] = table1_original['A→B Confidence'].apply(lambda x: f"{x:.1%}")
table1_original['B→A Confidence'] = table1_original['B→A Confidence'].apply(lambda x: f"{x:.1%}")
table1_original['Association Strength'] = table1_original['Association Strength'].apply(lambda x: f"{x:.2f}x")
table1_original['Product A Total Sales'] = table1_original['Product A Total Sales'].apply(lambda x: f"{x:,}")
table1_original['Product B Total Sales'] = table1_original['Product B Total Sales'].apply(lambda x: f"{x:,}")

print(f"✓ Table 1 formatted: {len(table1_original):,} rows")

# ============================================================================
# CREATE BUSINESS-FRIENDLY TABLE 2: FILTERED RESULTS
# ============================================================================
print("Creating Table 2: Filtered Results...")

table2_filtered = filtered_results.copy()
table2_filtered = table2_filtered.rename(columns={
    'item_a': 'Product A',
    'item_b': 'Product B',
    'pair_count': 'Times Purchased Together',
    'support': 'Market Frequency',
    'confidence_a_to_b': 'A→B Confidence',
    'confidence_b_to_a': 'B→A Confidence',
    'lift': 'Association Strength',
    'item_a_count': 'Product A Total Sales',
    'item_b_count': 'Product B Total Sales'
})

# Format numbers for business presentation
table2_filtered['Times Purchased Together'] = table2_filtered['Times Purchased Together'].apply(lambda x: f"{x:,}")
table2_filtered['Market Frequency'] = table2_filtered['Market Frequency'].apply(lambda x: f"{x:.4f}")
table2_filtered['A→B Confidence'] = table2_filtered['A→B Confidence'].apply(lambda x: f"{x:.1%}")
table2_filtered['B→A Confidence'] = table2_filtered['B→A Confidence'].apply(lambda x: f"{x:.1%}")
table2_filtered['Association Strength'] = table2_filtered['Association Strength'].apply(lambda x: f"{x:.2f}x")
table2_filtered['Product A Total Sales'] = table2_filtered['Product A Total Sales'].apply(lambda x: f"{x:,}")
table2_filtered['Product B Total Sales'] = table2_filtered['Product B Total Sales'].apply(lambda x: f"{x:,}")

print(f"✓ Table 2 formatted: {len(table2_filtered):,} rows")

# ============================================================================
# SAVE BOTH TABLES
# ============================================================================
print("\nSaving business tables...")

# Save Table 1: Original Results
table1_file = f"{BASE_DIR}/Table1_Original_Results.csv"
table1_original.to_csv(table1_file, index=False)
print(f"✓ Saved Table 1 to: Table1_Original_Results.csv")

# Save Table 2: Filtered Results
table2_file = f"{BASE_DIR}/Table2_Filtered_Results.csv"
table2_filtered.to_csv(table2_file, index=False)
print(f"✓ Saved Table 2 to: Table2_Filtered_Results.csv")

# Save both tables in one Excel file with separate sheets
excel_file = f"{BASE_DIR}/Market_Basket_Results_Both_Tables.xlsx"
with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    table1_original.to_excel(writer, sheet_name='Original Results', index=False)
    table2_filtered.to_excel(writer, sheet_name='Filtered Results', index=False)

print(f"✓ Saved combined Excel file: Market_Basket_Results_Both_Tables.xlsx")

# ============================================================================
# SHOW PREVIEW OF BOTH TABLES
# ============================================================================
print(f"\n" + "="*80)
print("📊 TABLE 1: ORIGINAL RESULTS (Including Cups)")
print("="*80)
print(table1_original.head(10).to_string(index=False))

print(f"\n" + "="*80)
print("📊 TABLE 2: FILTERED RESULTS (Excluding Cups)")
print("="*80)
print(table2_filtered.head(10).to_string(index=False))

print(f"\n✅ COMPLETE!")
print(f"📁 Three files created:")
print(f"   • Table1_Original_Results.csv")
print(f"   • Table2_Filtered_Results.csv")
print(f"   • Market_Basket_Results_Both_Tables.xlsx (both sheets)")

# Show column definitions for customer reference
print(f"\n📋 COLUMN DEFINITIONS:")
print("• Product A & Product B: The two items frequently bought together")
print("• Times Purchased Together: Number of transactions containing both items")
print("• Market Frequency: How often this pair appears relative to all transactions")
print("• A→B Confidence: If customer buys Product A, probability of also buying Product B")
print("• B→A Confidence: If customer buys Product B, probability of also buying Product A")
print("• Association Strength: How much more likely items are bought together vs independently")
print("• Product A/B Total Sales: Total number of transactions each product appears in")

=== CREATING BUSINESS-FRIENDLY TABLES ===

Loading results...
✓ Original results: 7,253 pairs
✓ Filtered results: 6,872 pairs

Creating Table 1: Original Results...
✓ Table 1 formatted: 7,253 rows
Creating Table 2: Filtered Results...
✓ Table 2 formatted: 6,872 rows

Saving business tables...
✓ Saved Table 1 to: Table1_Original_Results.csv
✓ Saved Table 2 to: Table2_Filtered_Results.csv
✓ Saved combined Excel file: Market_Basket_Results_Both_Tables.xlsx

📊 TABLE 1: ORIGINAL RESULTS (Including Cups)
                  Product A                    Product B Times Purchased Together Market Frequency Product A Total Sales Product B Total Sales A→B Confidence B→A Confidence Association Strength
           Medium Pepsi Cup                        Pepsi                   24,055           0.1438                44,806                25,135          53.7%          95.7%                3.57x
Aquafina Bottled Water 20oz             Medium Pepsi Cup                   14,860           0.0888          